# Using the New Rashomon-Set API

This tutorial is a small, self-contained demo of the redesigned Rashomon-set
search in `src.interval_utils.compute_rashomon_set`. The new API gives you
three things the old one didn't:

1. **Choose the certification method.** The optimization loop that grows the
   parameter box always uses fast IBP, but the *reported* certificates can be
   computed with any registered verification method (`IBP`, `CROWN`,
   `alpha-CROWN`, ...) via `certification_method`.
2. **Certify against input regions, not just points.** Pass
   `has_input_intervals=True` and a dataset of `(x_l, x_u, y)` triples to
   certify the Rashomon set against state *regions* (e.g. sensor noise),
   instead of single observations.
3. **A single, explicit `AccuracyRequirement`.** One dataclass replaces the
   old scattered `min_acc_limit` / `min_soft_acc_limit` / `min_hard_acc_limit`
   / `soft_acc_temperature` / `aggregation` / `multi_label_soft_metric`
   parameters, and an explicit `group_by` function replaces the old
   `task_labels` list for per-group accuracy targets.

We'll train a tiny toy classifier and walk through each of these in turn.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "core").is_dir() and (candidate / "tutorials").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current directory.")


REPO_ROOT = find_repo_root()
CORE_PATH = REPO_ROOT / "core"
if str(CORE_PATH) not in sys.path:
    sys.path.insert(0, str(CORE_PATH))

from src.interval_utils import compute_rashomon_set  # noqa: E402
from src.rashomon_spec import AccuracyRequirement, RashomonResult  # noqa: E402

SEED = 0
torch.manual_seed(SEED)
REPO_ROOT


PosixPath('/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning')

## 1. A Tiny Toy Classifier

A 2-class dataset, linearly separable by `x0 + x1 - x2 > 0`, and a small
`Linear -> ReLU -> Linear` network trained to (near-)perfectly classify it.
The Rashomon set is the set of weight perturbations around this trained
network that still meet an accuracy requirement we'll specify below.

In [2]:
N = 200
X = torch.randn(N, 3)
y = ((X[:, 0] + X[:, 1] - X[:, 2]) > 0).long()

model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
for _ in range(300):
    optimizer.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    optimizer.step()
model.eval()

with torch.no_grad():
    train_acc = (model(X).argmax(dim=1) == y).float().mean().item()
print(f"final loss: {loss.item():.4f}, train accuracy: {train_acc:.4f}")


def summarize(result: RashomonResult, label: str) -> None:
    """Pretty-print the last checkpoint's certificates from a RashomonResult."""
    print(f"{label}: {len(result.bounded_models)} checkpoint(s) returned")
    for cert in result.certificates[-1]:
        group_str = f"group={cert.group}" if cert.group is not None else "global"
        print(f"  {group_str:<10} soft_acc>={cert.min_soft_acc:.3f}  hard_acc>={cert.min_hard_acc:.3f}")


final loss: 0.0224, train accuracy: 0.9950


## 2. `AccuracyRequirement`: One Dataclass for All the Accuracy Knobs

`AccuracyRequirement` bundles everything that used to be five or six
separate, easy-to-misalign parameters:

- `soft_min` / `hard_min`: minimum *soft* (differentiable surrogate, used as
  the Lagrangian constraint) and *hard* (strict, certified) accuracy. Each
  can be a single float shared by every group, or a `dict[group_id, float]`
  for per-group targets. `hard_min` defaults to `soft_min` if omitted.
- `soft_metric` / `soft_temperature`: which differentiable surrogate to use
  (`"soft_accuracy"` or `"accuracy_margin"`) and its softmax temperature.
- `aggregation`: how per-sample correctness is reduced within a group
  (`"mean"` or the worst-case `"min"`).

`.resolve(group)` looks up the right `(soft_min, hard_min)` pair for a given
group id - this is exactly what `compute_rashomon_set` calls internally.

In [3]:
shared = AccuracyRequirement(soft_min=0.9, hard_min=0.95)
print("shared limits, any group:", shared.resolve(0), shared.resolve(1))

per_group = AccuracyRequirement(soft_min={0: 0.5, 1: 0.95}, hard_min={0: 0.6, 1: 1.0})
print("per-group limits, group 0:", per_group.resolve(0))
print("per-group limits, group 1:", per_group.resolve(1))

defaulted = AccuracyRequirement(soft_min=0.8)  # hard_min omitted -> defaults to soft_min
print("hard_min defaults to soft_min:", defaulted.resolve(0))


shared limits, any group: (0.9, 0.95) (0.9, 0.95)
per-group limits, group 0: (0.5, 0.6)
per-group limits, group 1: (0.95, 1.0)
hard_min defaults to soft_min: (0.8, 0.8)


## 3. Computing a Rashomon Set

By default, `compute_rashomon_set` treats the whole dataset as a single
global group: it grows a box around the trained weights as large as possible
while keeping the certified accuracy above `accuracy.hard_min` (and the
differentiable surrogate above `accuracy.soft_min`, used to drive the
optimization). The optimization loop is capped at a small `n_iters` here so
the tutorial runs quickly - in practice you'd use the (much larger) default
of 2000.

In [4]:
dataset = torch.utils.data.TensorDataset(X, y)

result = compute_rashomon_set(
    model, dataset,
    AccuracyRequirement(soft_min=0.85, hard_min=0.85),
    batch_size=N, certificate_samples=N, n_iters=150,
)
summarize(result, "single global group")


Initial acc constraint violation (group=None): -0.1451 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with limits: {None: (0.85, 0.85)}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.995063841342926, min_hard_acc=0.9950000047683716)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.20, obj=0.000, min_soft_acc=0.995]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.42, obj=0.004, min_soft_acc=0.995]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.66, obj=0.008, min_soft_acc=0.995]

...

 96%|█████████▌| 144/150 [00:01<00:00, 131.35it/s, size=20.43, obj=0.409, min_soft_acc=0.647]

 96%|█████████▌| 144/150 [00:01<00:00, 131.35it/s, size=20.42, obj=0.409, min_soft_acc=0.647]

 96%|█████████▌| 144/150 [00:01<00:00, 131.35it/s, size=20.42, obj=0.408, min_soft_acc=0.647]

100%|██████████| 150/150 [00:01<00:00, 124.41it/s, size=20.42, obj=0.408, min_soft_acc=0.647]

Final bbox:  Obj=0.41,  Size=20.42,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.647564709186554, min_hard_acc=0.6499999761581421)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
single global group: 1 checkpoint(s) returned
  global     soft_acc>=0.648  hard_acc>=0.650


## 4. Per-Group Accuracy Requirements via `group_by`

`group_by` replaces the old `task_labels` list. It's a function applied to
each minibatch's `y` that returns an integer group id per sample - each
unique id gets its own Lagrangian constraint with its own
`accuracy.resolve(group)` limits. For single-label classification,
`group_by=lambda y: y` recovers "one constraint per class", but `group_by`
works just as well on multi-hot admissible-action-set targets (anything that
can be turned into an integer group id).

Here we ask for a *looser* requirement on class 0 and a *stricter* one on
class 1, simulating a setting where one class is more safety-critical than
the other.

In [5]:
result_grouped = compute_rashomon_set(
    model, dataset,
    AccuracyRequirement(soft_min={0: 0.5, 1: 0.95}, hard_min={0: 0.6, 1: 1.0}),
    batch_size=N, certificate_samples=N, n_iters=150,
    group_by=lambda y: y,
)
summarize(result_grouped, "per-class group_by")


Initial acc constraint violation (group=0): -0.5000 (Positive = violated)
Initial acc constraint violation (group=1): -0.0387 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with limits: {0: (0.5, 0.6), 1: (0.95, 1.0)}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=0, min_soft_acc=0.9999601244926453, min_hard_acc=1.0), RashomonCertificate(group=1, min_soft_acc=0.9887043833732605, min_hard_acc=0.9885057210922241)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.20, obj=0.000, min_soft_acc=0.989]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.42, obj=0.004, min_soft_acc=0.988]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.66, obj=0.008, min_soft_acc=0.988]

...

 95%|█████████▌| 143/150 [00:01<00:00, 106.05it/s, size=9.64, obj=0.193, min_soft_acc=0.885]

 95%|█████████▌| 143/150 [00:01<00:00, 106.05it/s, size=9.64, obj=0.193, min_soft_acc=0.885]

 95%|█████████▌| 143/150 [00:01<00:00, 106.05it/s, size=9.65, obj=0.193, min_soft_acc=0.885]

100%|██████████| 150/150 [00:01<00:00, 105.77it/s, size=9.65, obj=0.193, min_soft_acc=0.885]

Final bbox:  Obj=0.19,  Size=9.65,  Certificates=[RashomonCertificate(group=0, min_soft_acc=0.8304397463798523, min_hard_acc=0.8318583965301514), RashomonCertificate(group=1, min_soft_acc=0.8849310278892517, min_hard_acc=0.8850574493408203)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
per-class group_by: 1 checkpoint(s) returned
  group=0    soft_acc>=0.830  hard_acc>=0.832
  group=1    soft_acc>=0.885  hard_acc>=0.885


## 5. Certifying Against Input Regions, Not Just Points

Pass `has_input_intervals=True` together with a dataset that yields
`(x_l, x_u, y)` triples (instead of `(x, y)`) to certify the Rashomon set
against an input *region* around each sample - e.g. sensor noise or
discretization error - rather than the single observed point. Internally,
every `bound_forward` call inside the optimization and certification now
uses `(x_l, x_u)` instead of `(x, x)`.

Widening the input region can only make the certified accuracy go down or
stay the same (the model has to be correct on harder, perturbed inputs too),
so we expect a lower certificate than the point-only run above for the same
accuracy target.

In [6]:
eps = 0.02
X_l, X_u = X - eps, X + eps
interval_dataset = torch.utils.data.TensorDataset(X_l, X_u, y)

result_interval = compute_rashomon_set(
    model, interval_dataset,
    AccuracyRequirement(soft_min=0.85, hard_min=0.85),
    batch_size=N, certificate_samples=N, n_iters=150,
    has_input_intervals=True,
)
summarize(result_interval, f"certified against a +-{eps} input box")


Initial acc constraint violation (group=None): -0.1323 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with limits: {None: (0.85, 0.85)}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.9822763800621033, min_hard_acc=0.9800000190734863)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.20, obj=0.000, min_soft_acc=0.982]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.42, obj=0.004, min_soft_acc=0.980]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.66, obj=0.008, min_soft_acc=0.977]

...

 96%|█████████▌| 144/150 [00:01<00:00, 132.91it/s, size=18.53, obj=0.371, min_soft_acc=0.642]

 96%|█████████▌| 144/150 [00:01<00:00, 132.91it/s, size=18.51, obj=0.371, min_soft_acc=0.642]

 96%|█████████▌| 144/150 [00:01<00:00, 132.91it/s, size=18.49, obj=0.370, min_soft_acc=0.643]

100%|██████████| 150/150 [00:01<00:00, 115.93it/s, size=18.49, obj=0.370, min_soft_acc=0.643]

Final bbox:  Obj=0.37,  Size=18.49,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.6430986523628235, min_hard_acc=0.6449999809265137)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
certified against a +-0.02 input box: 1 checkpoint(s) returned
  global     soft_acc>=0.643  hard_acc>=0.645


## 6. Choosing the Certification Method

The Lagrangian loop that grows the parameter box always uses `IBP` - it's
fast, and the loop runs hundreds to thousands of iterations. But the
*reported* certificates don't have to use the same method: pass
`certification_method` (and optionally `certification_method_kwargs`) to
rebuild a `BoundedModel` from each checkpoint's `(param_l, param_u)` using
any registered verification method, and certify against that instead. This
reuses `build_bounded_model` from `src.verification.api` under the hood.

Different methods make different soundness/tightness trade-offs, so the
reported certificate can differ between methods even though the optimized
box itself is identical.

In [7]:
for method in ["IBP", "CROWN"]:
    result_method = compute_rashomon_set(
        model, dataset,
        AccuracyRequirement(soft_min=0.85, hard_min=0.85),
        batch_size=N, certificate_samples=N, n_iters=150,
        certification_method=method,
    )
    summarize(result_method, method)


Initial acc constraint violation (group=None): -0.1451 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with limits: {None: (0.85, 0.85)}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.995063841342926, min_hard_acc=0.9950000047683716)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.20, obj=0.000, min_soft_acc=0.995]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.42, obj=0.004, min_soft_acc=0.995]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.66, obj=0.008, min_soft_acc=0.995]

...

 99%|█████████▊| 148/150 [00:01<00:00, 147.57it/s, size=20.43, obj=0.409, min_soft_acc=0.647]

 99%|█████████▊| 148/150 [00:01<00:00, 147.57it/s, size=20.42, obj=0.409, min_soft_acc=0.647]

 99%|█████████▊| 148/150 [00:01<00:00, 147.57it/s, size=20.42, obj=0.408, min_soft_acc=0.647]

100%|██████████| 150/150 [00:01<00:00, 145.38it/s, size=20.42, obj=0.408, min_soft_acc=0.647]

Final bbox:  Obj=0.41,  Size=20.42,  Certificates=[RashomonCertificate(group=None, min_soft_acc=0.647564709186554, min_hard_acc=0.6499999761581421)]
Computing final certificates over 200 samples using CROWN
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
CROWN: 1 checkpoint(s) returned
  global     soft_acc>=0.563  hard_acc>=0.565


## Notes

- `IntervalTrainer.compute_rashomon_set` (in `src.trainer.IntervalTrainer`)
  is a thin wrapper around this same function for continual-learning
  pipelines: it resolves a trainer-level `AccuracyRequirement` (with its own
  `min_acc_increment` policy) and forwards `group_by`, `has_input_intervals`,
  `certification_method`, and `certification_method_kwargs` straight through.
- `RashomonResult.certificates` is always a list (one entry per checkpoint)
  of lists (one `RashomonCertificate` per group) - a uniform shape regardless
  of whether `group_by` was given, replacing the old ad hoc tuple-of-lists
  return value.
- `checkpoint=N` in `compute_rashomon_set` records a deep copy of the
  optimization-time box every `N` iterations (plus the final one); each
  checkpoint gets its own certificates, which is useful for tracking how the
  Rashomon set's certified accuracy evolves as the box grows.